# Surface Code Simulation — Exploration

This notebook exercises `surface_code_sim.py` to study how logical error rates behave under:

- Varying code distance  
- Isotropic Pauli (depolarizing) gate noise  
- Independent measurement / reset noise  
- Different numbers of syndrome extraction rounds  

**Decoder:** PyMatching (MWPM)

**Circuit model:** Stim rotated surface code, logical idle experiment  
(prepare logical |0⟩, do $r$ rounds of syndrome extraction, measure logical Z)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from surface_code_sim import (
    CodeType,
    ErrorModel,
    PyMatchingDecoder,
    SimulationResult,
    SurfaceCodeSimulator,
    threshold_sweep,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
SEED = 42

---
## 1. Single-shot sanity check

Run one logical idle experiment with distance-3, symmetric noise at $p = 0.01$.

In [ ]:
sim = SurfaceCodeSimulator(distance=3)

em = ErrorModel.symmetric(p=0.01)
result = sim.run(em, rounds=3, shots=5_000, seed=SEED)
print(result)

# Inspect the noiseless and noisy circuits
noiseless = sim.build_circuit(ErrorModel(0, 0), rounds=3)
noisy     = sim.build_circuit(em, rounds=3)
print(f"\nCircuit stats  |  noiseless: {len(noiseless)} instructions, "
      f"noisy: {len(noisy)} instructions")

### 1a. View the detector error model

The DEM is what PyMatching uses to build its matching graph.

In [ ]:
dem = noisy.detector_error_model(decompose_errors=True)
print(str(dem)[:2000], "..." if len(str(dem)) > 2000 else "")

---
## 2. Logical error rate vs. physical error rate (single distance)

Sweep $p_{\text{phys}}$ for $d=5$ with symmetric noise ($p_{\text{meas}} = p_{\text{phys}}$).

In [ ]:
d = 5
p_values = np.logspace(-3, -0.7, 20)
sim5 = SurfaceCodeSimulator(distance=d)

results_d5 = sim5.sweep_p(
    p_values, rounds=d, shots=8_000, p_meas_factor=1.0, seed=SEED
)

p_arr  = np.array([r.error_model.p_phys for r in results_d5])
ler    = np.array([r.logical_error_rate for r in results_d5])
ler_se = np.array([r.logical_error_rate_se for r in results_d5])

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(p_arr, ler, yerr=ler_se, fmt='o-', capsize=3, label=f'd={d}')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Physical error rate $p$')
ax.set_ylabel('Logical error rate')
ax.set_title('Logical error rate vs. physical error rate (symmetric noise)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Threshold sweep — multiple distances

The curves for different $d$ cross near the **threshold** $p_\text{th}$.  
Below threshold the larger code wins; above it, the smaller code wins.

In [ ]:
distances = [3, 5, 7]
p_values  = np.logspace(-3, -0.5, 25)

sweep = threshold_sweep(
    distances=distances,
    p_values=p_values,
    shots=8_000,
    p_meas_factor=1.0,
    code_type=CodeType.ROTATED_Z,
    seed=SEED,
)

fig, ax = plt.subplots(figsize=(8, 5))
for d, results in sweep.items():
    p_arr  = np.array([r.error_model.p_phys for r in results])
    ler    = np.array([r.logical_error_rate  for r in results])
    ler_se = np.array([r.logical_error_rate_se for r in results])
    ax.errorbar(p_arr, ler, yerr=ler_se, fmt='o-', capsize=3, label=f'd={d}')

ax.axvline(0.01, color='k', ls='--', alpha=0.4, label='~1% threshold guide')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Physical error rate $p$')
ax.set_ylabel('Logical error rate')
ax.set_title('Threshold sweep — rotated surface code (symmetric noise)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Measurement noise vs. gate noise

Fix $p_{\text{phys}} = 0.005$ and vary $p_{\text{meas}}$ independently to see how measurement noise alone degrades the logical error rate.

In [ ]:
p_phys_fixed = 0.005
p_meas_values = np.logspace(-3, -0.5, 20)

sim5 = SurfaceCodeSimulator(distance=5)
results_meas = [
    sim5.run(ErrorModel(p_phys=p_phys_fixed, p_meas=pm), rounds=5, shots=8_000, seed=SEED)
    for pm in p_meas_values
]

# Also fix p_meas=0.005 and vary p_phys
p_meas_fixed = 0.005
p_phys_values = np.logspace(-3, -0.5, 20)
results_gate = [
    sim5.run(ErrorModel(p_phys=pp, p_meas=p_meas_fixed), rounds=5, shots=8_000, seed=SEED)
    for pp in p_phys_values
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(
    p_meas_values,
    [r.logical_error_rate for r in results_meas],
    yerr=[r.logical_error_rate_se for r in results_meas],
    fmt='s-', capsize=3, label=f'vary $p_{{meas}}$, $p_{{phys}}={p_phys_fixed}$ fixed'
)
ax.errorbar(
    p_phys_values,
    [r.logical_error_rate for r in results_gate],
    yerr=[r.logical_error_rate_se for r in results_gate],
    fmt='o-', capsize=3, label=f'vary $p_{{phys}}$, $p_{{meas}}={p_meas_fixed}$ fixed'
)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Varied error rate')
ax.set_ylabel('Logical error rate  (d=5, r=5)')
ax.set_title('Gate noise vs. measurement noise sensitivity')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Effect of syndrome extraction rounds

For a fixed error rate, more rounds = more opportunities for errors but also better temporal redundancy.  The logical error rate per *round* should approach a constant.

In [ ]:
em_low  = ErrorModel.symmetric(0.003)
em_high = ErrorModel.symmetric(0.008)
round_values = list(range(1, 16))

sim5 = SurfaceCodeSimulator(distance=5)
res_low  = sim5.sweep_rounds(round_values, em_low,  shots=8_000, seed=SEED)
res_high = sim5.sweep_rounds(round_values, em_high, shots=8_000, seed=SEED)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, res, em, label in [
    (axes[0], res_low,  em_low,  'Cumulative LER'),
    (axes[1], res_high, em_high, 'Cumulative LER'),
]:
    rs  = np.array([r.rounds for r in res])
    ler = np.array([r.logical_error_rate for r in res])
    se  = np.array([r.logical_error_rate_se for r in res])
    ax.errorbar(rs, ler, yerr=se, fmt='o-', capsize=3)
    ax.set_xlabel('Syndrome rounds')
    ax.set_ylabel(label)
    ax.set_title(f'd=5, p={em.p_phys}')

plt.suptitle('Logical error rate vs. syndrome extraction rounds', y=1.02)
plt.tight_layout()
plt.show()

# LER per round
fig, ax = plt.subplots(figsize=(7, 4))
for res, em in [(res_low, em_low), (res_high, em_high)]:
    rs  = np.array([r.rounds for r in res])
    ler = np.array([r.logical_error_rate for r in res])
    ax.plot(rs, ler / rs, 'o-', label=f'p={em.p_phys}')
ax.set_xlabel('Syndrome rounds')
ax.set_ylabel('LER per round')
ax.set_title('d=5  —  LER per round converges as rounds grow')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Noise asymmetry — gate-dominated vs. measurement-dominated

Compare three noise regimes at the same total effective error rate.

In [ ]:
p_values = np.logspace(-3, -0.5, 20)
distances_asym = [3, 5]

def run_regime(label, meas_factor):
    return label, threshold_sweep(
        distances=distances_asym,
        p_values=p_values,
        shots=6_000,
        p_meas_factor=meas_factor,
        seed=SEED,
    )

regimes = [
    run_regime('Symmetric  (p_meas = p_phys)',   meas_factor=1.0),
    run_regime('Gate-dominated (p_meas = 0.1·p)', meas_factor=0.1),
    run_regime('Meas-dominated (p_meas = 10·p)',  meas_factor=10.0),
]

fig, axes = plt.subplots(1, len(regimes), figsize=(15, 5), sharey=True)
for ax, (label, sweep) in zip(axes, regimes):
    for d, results in sweep.items():
        p_arr  = np.array([r.error_model.p_phys for r in results])
        ler    = np.array([r.logical_error_rate  for r in results])
        ler_se = np.array([r.logical_error_rate_se for r in results])
        ax.errorbar(p_arr, ler, yerr=ler_se, fmt='o-', capsize=2, label=f'd={d}')
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('$p_{phys}$')
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=9)

axes[0].set_ylabel('Logical error rate')
plt.suptitle('Noise asymmetry comparison')
plt.tight_layout()
plt.show()

---
## 7. Code type comparison — rotated vs. unrotated

Rotated codes use ~half the physical qubits for the same distance.  
Here we compare them at the same distance and noise model.

In [ ]:
p_values_ct = np.logspace(-3, -0.5, 20)

code_types = {
    'Rotated Z-memory':   CodeType.ROTATED_Z,
    'Unrotated Z-memory': CodeType.UNROTATED_Z,
}

fig, ax = plt.subplots(figsize=(8, 5))
for label, ct in code_types.items():
    sim_ct = SurfaceCodeSimulator(distance=5, code_type=ct)
    res    = sim_ct.sweep_p(p_values_ct, rounds=5, shots=6_000, seed=SEED)
    p_arr  = np.array([r.error_model.p_phys for r in res])
    ler    = np.array([r.logical_error_rate  for r in res])
    ler_se = np.array([r.logical_error_rate_se for r in res])
    ax.errorbar(p_arr, ler, yerr=ler_se, fmt='o-', capsize=3, label=label)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Physical error rate $p$')
ax.set_ylabel('Logical error rate  (d=5, r=5)')
ax.set_title('Rotated vs. unrotated surface code (symmetric noise)')
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. Logical error rate scaling with distance (below threshold)

Below threshold, the logical error rate should scale as  
$$p_L \approx A \left(\frac{p}{p_{\text{th}}}\right)^{\lfloor(d+1)/2\rfloor}$$
A log-log plot of $p_L$ vs $d$ should be roughly linear.

In [ ]:
p_below = 0.003   # well below threshold
em_below = ErrorModel.symmetric(p_below)
distances_scale = [3, 5, 7, 9]

scale_results = []
for d in distances_scale:
    sim_d = SurfaceCodeSimulator(distance=d)
    r = sim_d.run(em_below, rounds=d, shots=20_000, seed=SEED)
    scale_results.append(r)
    print(r)

fig, ax = plt.subplots(figsize=(7, 4))
ds  = np.array([r.distance for r in scale_results])
ler = np.array([r.logical_error_rate for r in scale_results])
se  = np.array([r.logical_error_rate_se for r in scale_results])

ax.errorbar(ds, ler, yerr=se, fmt='o-', capsize=4)
# Fit a power law in d
valid = ler > 0
if valid.sum() >= 2:
    coeffs = np.polyfit(ds[valid], np.log(ler[valid]), 1)
    d_fit  = np.linspace(ds.min(), ds.max(), 100)
    ax.plot(d_fit, np.exp(np.polyval(coeffs, d_fit)), 'k--', alpha=0.5,
            label=f'Exponential fit  e^({coeffs[0]:.2f}·d)')
    ax.legend()

ax.set_yscale('log')
ax.set_xlabel('Code distance $d$')
ax.set_ylabel('Logical error rate')
ax.set_title(f'LER vs. distance at $p = {p_below}$ (below threshold)')
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.tight_layout()
plt.show()

---
## 9. Custom decoder hook

You can plug in any decoder that implements the `Decoder` interface.  
Here we show how to wrap a simple majority-vote decoder as an example skeleton.

In [ ]:
from surface_code_sim import Decoder
import stim

class TrivialDecoder(Decoder):
    """Always predicts no logical error — useful as a baseline."""

    def setup(self, circuit: stim.Circuit) -> None:
        dem = circuit.detector_error_model()
        self._num_obs = dem.num_observables

    def decode_batch(self, detection_events: np.ndarray) -> np.ndarray:
        shots = detection_events.shape[0]
        return np.zeros((shots, self._num_obs), dtype=bool)


# Compare PyMatching vs trivial at p = 0.01
em_cmp = ErrorModel.symmetric(0.01)
sim_cmp = SurfaceCodeSimulator(distance=5)

r_mwpm    = sim_cmp.run(em_cmp, rounds=5, shots=10_000, decoder=PyMatchingDecoder(), seed=SEED)
r_trivial = sim_cmp.run(em_cmp, rounds=5, shots=10_000, decoder=TrivialDecoder(),    seed=SEED)

print(f"PyMatching:      {r_mwpm}")
print(f"Trivial decoder: {r_trivial}")
print(f"\nSpeedup factor in LER: {r_trivial.logical_error_rate / r_mwpm.logical_error_rate:.1f}×")

---
## 10. High-statistics run near threshold

Get reliable statistics near the threshold region with more shots.

In [ ]:
p_near_thresh = np.linspace(0.005, 0.025, 12)
distances_hi  = [3, 5, 7]

sweep_hi = threshold_sweep(
    distances=distances_hi,
    p_values=p_near_thresh,
    shots=30_000,
    p_meas_factor=1.0,
    seed=SEED,
)

fig, ax = plt.subplots(figsize=(8, 5))
for d, results in sweep_hi.items():
    p_arr  = np.array([r.error_model.p_phys for r in results])
    ler    = np.array([r.logical_error_rate  for r in results])
    ler_se = np.array([r.logical_error_rate_se for r in results])
    ax.errorbar(p_arr * 100, ler, yerr=ler_se, fmt='o-', capsize=3, label=f'd={d}')

ax.set_yscale('log')
ax.set_xlabel('Physical error rate $p$ (%)')
ax.set_ylabel('Logical error rate')
ax.set_title('High-statistics threshold region (30k shots, symmetric noise)')
ax.legend()
plt.tight_layout()
plt.show()